# Embed Inside Airbnb London reviews (GPU)

Bulk-embed ~2.24M reviews with `BAAI/bge-small-en-v1.5` on a Colab GPU.

Query-time vectors are produced locally with `fastembed` using the **same model**, so the vectors are compatible and no schema change or runtime API key is needed.

**Before running:** `Runtime -> Change runtime type -> Hardware accelerator: GPU (T4)`, then `Runtime -> Run all`.

Output shards are saved to Google Drive at `MyDrive/review_embeddings/`.

In [ ]:
!pip -q install sentence-transformers

import os, shutil, urllib.request
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

try:
    from google.colab import drive, files
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception as error:  # noqa: BLE001
    print('not in Colab (download only):', error)
    IN_COLAB = False

In [ ]:
REVIEWS_URL = (
    'https://data.insideairbnb.com/united-kingdom/england/london/'
    '2026-06-19/data/reviews.csv.gz'
)
LOCAL_CSV = '/content/reviews.csv.gz'
OUT_DIR = '/content/drive/MyDrive/review_embeddings' if IN_COLAB else '/content/review_embeddings'

if not os.path.exists(LOCAL_CSV):
    print('downloading', REVIEWS_URL)
    urllib.request.urlretrieve(REVIEWS_URL, LOCAL_CSV)
print('bytes:', os.path.getsize(LOCAL_CSV))

In [ ]:
frame = pd.read_csv(LOCAL_CSV, usecols=['id', 'listing_id', 'comments'])
frame = frame[frame['comments'].notna()].copy()
frame['content'] = frame['comments'].astype(str).str.slice(0, 1000)
frame = frame[frame['content'].str.len() > 40]
print('rows:', len(frame))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
model = SentenceTransformer('BAAI/bge-small-en-v1.5', device=device)

os.makedirs(OUT_DIR, exist_ok=True)
SHARD = 200_000
for start in range(0, len(frame), SHARD):
    path = f'{OUT_DIR}/shard_{start:09d}.npz'
    if os.path.exists(path):
        print('skip', path)
        continue
    part = frame.iloc[start:start + SHARD]
    vectors = model.encode(
        part['content'].tolist(),
        batch_size=512,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float16)
    np.savez_compressed(
        path,
        review_id=part['id'].to_numpy(np.int64),
        listing_id=part['listing_id'].to_numpy(np.int64),
        embedding=vectors,
    )
    print('wrote', path, vectors.shape)
print('done')

## Get the embeddings out

Option A (recommended): in Google Drive, right-click the `review_embeddings` folder -> **Share** -> **Anyone with the link**, and give the link to the agent (it will fetch with `gdown`).

Option B: run the cell below to zip and download `review_embeddings.zip` (~1.6-1.8 GB).

In [ ]:
zip_base = '/content/review_embeddings'
shutil.make_archive(zip_base, 'zip', OUT_DIR)
zip_path = zip_base + '.zip'
print('zip:', zip_path, round(os.path.getsize(zip_path) / 1e9, 2), 'GB')
print('shards:', sorted(os.listdir(OUT_DIR)))

# Uncomment to download in the browser:
# files.download(zip_path)